In [ ]:
import pandas as pd
import simplekml
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import os
from matplotlib.colors import to_rgb, to_hex

# Parameter for the time window in minutes
time_window_minutes = 60
buffer_time = 2
morning_hour = 9
# Read the CSV file
#os.chdir('c:\\Users\\user01\\Documents\\Github\\MBRP')
data = pd.read_csv('data\\gps_v1.csv')

# Convert timestamp to datetime for easier manipulation
data['timestamp'] = pd.to_datetime(data['timestamp'], format='%Y-%m-%d %H:%M:%S.%f')

# data.timestamp = data.orientation_quaternion_raw_z
# data.timestamp = pd.to_datetime(d)
# data.individual_local_identifier = data.sensor_type_id


In [68]:
# Group the data by 'group_id'
grouped = data.groupby('group_id')

# Define distinct base colors for each group
# base_colors = [
#     (1.0, 0.9, 0.6),  # Light Yellow
#     (1.0, 0.6, 0.6),  # Light Red
#     (0.6, 1.0, 0.6),  # Light Green
#     (0.6, 0.6, 1.0),  # Light Blue
#     (1.0, 1.0, 0.6),  # Light Lime
#     (1.0, 0.6, 1.0),  # Light Magenta
#     (0.3, 0.7, 0.7),  # Teal
#     (1.0, 0.8, 0.6),  # Light Orange
#     (0.8, 0.6, 1.0),  # Light Purple
#     (0.9, 0.9, 0.9),  # Light Grey
#     (0.9, 0.7, 0.5)   # Light Brown
# ]
# # Function to generate a gradient of colors
# def generate_gradient(base_color, num_colors):
#     return [mcolors.to_hex((base_color[0] * (1 - i / num_colors), 
#                             base_color[1] * (1 - i / num_colors), 
#                             base_color[2] * (1 - i / num_colors))) for i in range(num_colors)]

# # Replace the gradient_colors generation with the base_colors dictionary
base_colors = {
    "Maroon": "#800000",        # Maroon
    "Chartreuse": "#7FFF00",         # Chartreuse
    "Bronze": "#CD7F32",     # Bronze
    "Emerald": "#50C878",       # Emerald
    "Lilac": "#C8A2C8",          # Lilac
    "Copper": "#B87333",         # Copper
    "Magenta": "#FF00FF",      # Magenta
    "LapisSplinter": "#87CEFA",      # LapisSplinter
    "Lapis": "#26619C",       # Lapis
    "Periwinkle": "#CCCCFF"             # Periwinkle
}


# Function to generate a gradient of colors

def generate_gradient(base_color, num_colors):
    # Convert hex color to RGB using mcolors
    base_color_rgb = to_rgb(base_color)
    
    # Generate gradient in RGB
    gradient_rgb = [
        (
            base_color_rgb[0] * (1 - i / num_colors),
            base_color_rgb[1] * (1 - i / num_colors),
            base_color_rgb[2] * (1 - i / num_colors)
        )
        for i in range(num_colors)
    ]
    
    # Convert RGB back to hex using mcolors
    gradient_hex = [to_hex(color) for color in gradient_rgb]
    
    return gradient_hex

# def generate_gradient(base_color, num_colors):
#     return [(base_color[0] * (1 - i / num_colors), 
#                             base_color[1] * (1 - i / num_colors), 
#                             base_color[2] * (1 - i / num_colors)) for i in range(num_colors)]

# # Map each individual to a color using the base_colors dictionary
# color_map = {individual: base_colors.get(individual, "#000000") for individual in individuals}  # Default to black if not found

# color_map

In [65]:
## full tracks
# Create a KMZ file for each group
for group_index, (group_id, group_data) in enumerate(grouped):
    kml = simplekml.Kml()
    
    # Sort data by individual and timestamp
    group_data = group_data.sort_values(by=['individual_local_identifier', 'timestamp'])
    
    # Get unique individuals in the group
    individuals = group_data['individual_local_identifier'].unique()
    num_individuals = len(individuals)
    
    # Generate a gradient of colors for the group
    #base_color = base_colors[group_index % len(base_colors)]
    base_color = base_colors[group_id]
    gradient_colors = generate_gradient(base_color, num_individuals)
    
    # Map each individual to a color
    color_map = {individual: gradient_colors[i] for i, individual in enumerate(individuals)}
    
    # Iterate over each individual
    for individual_id, individual_data in group_data.groupby('individual_local_identifier'):
        # Create a folder for each individual
        folder = kml.newfolder(name=individual_id)
        folder.visibility = 0  # Set folder visibility to 0 (hidden)
        
        # Initialize the start time for the first segment
        start_time = individual_data['timestamp'].min()
        end_time = start_time + timedelta(minutes=time_window_minutes)
        
        while start_time < individual_data['timestamp'].max():
            # Filter data within the current time window
            window_data = individual_data[(individual_data['timestamp'] >= start_time - timedelta(minutes=buffer_time)) & 
                                          (individual_data['timestamp'] <= end_time + timedelta(minutes=buffer_time))]
            
            if not window_data.empty:
                label = f"{individual_id} {start_time.strftime('%Y-%m-%d %H:%M:%S')}"
                line = folder.newlinestring(name=label)
                line.coords = list(zip(window_data['location.long'], window_data['location.lat']))
                
                # Set the color using the color map with alpha
                hex_color = color_map[individual_id]
                kml_color = 'b2' + hex_color[1:]  # Add alpha to hex color
                line.style.linestyle.color = kml_color
                line.style.linestyle.width = 5  # Set line width
                
                # Set the timespan for each line
                line.timespan.begin = start_time.strftime('%Y-%m-%dT%H:%M:%SZ')
                line.timespan.end = end_time.strftime('%Y-%m-%dT%H:%M:%SZ')
                
                # Set the visibility of the line to 0
                #line.visibility = 0
            
            # Move to the next time window
            start_time = end_time
            end_time = start_time + timedelta(minutes=time_window_minutes)
                
    # Save the KMZ file with the group_id as part of the filename
    kml.savekmz(f"plots\kmls\day\{group_id}.kmz")
    print(f"KMZ file for group {group_id} saved successfully.")

KMZ file for group Bronze saved successfully.
KMZ file for group Chartreuse saved successfully.
KMZ file for group Copper saved successfully.
KMZ file for group Emerald saved successfully.
KMZ file for group Lapis saved successfully.
KMZ file for group LapisSplinter saved successfully.
KMZ file for group Lilac saved successfully.
KMZ file for group Magenta saved successfully.
KMZ file for group Periwinkle saved successfully.


KeyError: 'WestMukenya'

In [ ]:
## Morning tracks
# Filter data to include only tracks before noon
# Group the data by 'group_id'

data_mor = data[data['timestamp'].dt.hour < morning_hour]

grouped = data_mor.groupby('group_id')

# Create a KMZ file for each group
for group_index, (group_id, group_data) in enumerate(grouped):
    kml = simplekml.Kml()
    
    # Sort data by individual and timestamp
    group_data = group_data.sort_values(by=['individual_local_identifier', 'timestamp'])
    
    # Get unique individuals in the group
    individuals = group_data['individual_local_identifier'].unique()
    num_individuals = len(individuals)
    
    # Generate a gradient of colors for the group
    base_color = base_colors[group_id]
    gradient_colors = generate_gradient(base_color, num_individuals)
    
    # Map each individual to a color
    color_map = {individual: gradient_colors[i] for i, individual in enumerate(individuals)}
    
    # Iterate over each individual
    for individual_id, individual_data in group_data.groupby('individual_local_identifier'):
        # Create a folder for each individual
        folder = kml.newfolder(name=individual_id)
        folder.visibility = 0  # Set folder visibility to 0 (hidden)
        
        # Initialize the start time for the first segment
        start_time = individual_data['timestamp'].min()
        end_time = start_time + timedelta(minutes=time_window_minutes)
        
        while start_time < individual_data['timestamp'].max():
            # Filter data within the current time window
            window_data = individual_data[(individual_data['timestamp'] >= start_time - timedelta(minutes=buffer_time)) & 
                                          (individual_data['timestamp'] <= end_time + timedelta(minutes=buffer_time))]
            
            if not window_data.empty:
                label = f"{individual_id} {start_time.strftime('%Y-%m-%d %H:%M:%S')}"
                line = folder.newlinestring(name=label)

                line.coords = list(zip(window_data['location.long'], window_data['location.lat']))
                
                # Set the color using the color map with alpha
                hex_color = color_map[individual_id]
                kml_color = 'b2' + hex_color[1:]  # Add alpha to hex color
                line.style.linestyle.color = kml_color
                line.style.linestyle.width = 5  # Set line width
                
                # Set the timespan for each line
                line.timespan.begin = start_time.strftime('%Y-%m-%dT%H:%M:%SZ')
                line.timespan.end = end_time.strftime('%Y-%m-%dT%H:%M:%SZ')
                
                # Set the visibility of the line to 0
                #line.visibility = 0
            
            # Move to the next time window
            start_time = end_time
            end_time = start_time + timedelta(minutes=time_window_minutes)
                
    # Save the KMZ file with the group_id as part of the filename
    kml.savekmz(f"morning\{group_id}.kmz")

In [79]:
# Get the current date and calculate the start of the last week
current_date = datetime.now().astimezone()  # Make current_date timezone-aware
last_week_start = current_date - timedelta(days=7)

# Filter data for the last week
data_last_week = data[(data['timestamp'] >= last_week_start) & (data['timestamp'] <= current_date)]
data_last_week

,sensor_type_id,individual_local_identifier,barometric_pressure,data_decoding_software,eobs_activity,eobs_activity_samples,eobs_battery_voltage,eobs_fix_battery_voltage,eobs_horizontal_accuracy_estimate,eobs_key_bin_checksum,...,visible,geometry,azimuth,speed,tag_local_identifier,group_id,sex,location.long,location.lat,plot_name
165801,653,24AA12_6P8Q,0,21,NaN,NaN,3865,3864,5.63,2460211642,...,True,36.9246762|0.3672884,-1.151871,0.003926,10372,Copper,m,36.924676,0.367288,Copper
165802,653,24AA12_6P8Q,0,21,NaN,NaN,3864,3864,19.97,4251178305,...,True,36.9246733|0.3672897,-0.539061,0.109484,10372,Copper,m,36.924673,0.367290,Copper
165803,653,24AA12_6P8Q,0,21,NaN,NaN,3864,3864,29.95,2132632304,...,True,36.9245878|0.3674336,2.424045,0.261835,10372,Copper,m,36.924588,0.367434,Copper
165804,653,24AA12_6P8Q,0,21,NaN,NaN,3864,3864,39.68,3653232622,...,True,36.9247513|0.367245,-0.957098,0.318887,10372,Copper,m,36.924751,0.367245,Copper
165805,653,24AA12_6P8Q,0,21,NaN,NaN,3864,3862,47.87,2401756760,...,True,36.9244776|0.3674391,2.184601,0.226879,10372,Copper,m,36.924478,0.367439,Copper
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6613229,653,24AE75_5F6G,0,21,NaN,NaN,4005,4000,13.57,4217377871,...,True,36.9123827|0.2992596,-2.004860,0.076057,15776,Maroon,m,36.912383,0.299260,Maroon
6613230,653,24AE75_5F6G,0,21,NaN,NaN,4005,4000,10.24,3403645543,...,True,36.9123077|0.2992246,2.928923,0.032970,15776,Maroon,m,36.912308,0.299225,Maroon
6613231,653,24AE75_5F6G,0,21,NaN,NaN,4006,3999,12.54,198128175,...,True,36.9123151|0.2991901,1.681560,0.156089,15776,Maroon,m,36.912315,0.299190,Maroon
6613232,653,24AE75_5F6G,0,21,NaN,NaN,4006,4002,18.43,3388909445,...,True,36.9124839|0.2991712,-1.700133,0.146821,15776,Maroon,m,36.912484,0.299171,Maroon


In [76]:
data['timestamp'] = pd.to_datetime(data['timestamp']).dt.tz_localize('UTC')


TypeError: Already tz-aware, use tz_convert to convert.

In [84]:
# Get the current date and calculate the start of the last week
current_date = datetime.now().astimezone()  # Make current_date timezone-aware
last_week_start = current_date - timedelta(days=7)

# Filter data for the last week
data_last_week = data[(data['timestamp'] >= last_week_start) & (data['timestamp'] <= current_date)]

# Group the data by 'group_id'
grouped = data_last_week.groupby('group_id')

# Create a KMZ file for the last week
kml = simplekml.Kml()

# Iterate over each group
for group_id, group_data in grouped:
    # Get the color for the group from base_colors
    group_color = base_colors.get(group_id, "#000000")  # Default to black if not found

    # Iterate over each individual in the group
    for individual_id, individual_data in group_data.groupby('individual_local_identifier'):
        # Create a folder for each individual
        folder = kml.newfolder(name=f"{individual_id} ({tag_local_identifier})")
        folder.visibility = 0  # Set folder visibility to 0 (hidden)

        tag_local_identifier = individual_data['tag_local_identifier'].iloc[0] if not individual_data['tag_local_identifier'].empty else 'Unknown'
        # Create a multiline for the individual
        line = folder.newlinestring(name=f"{individual_id} ({tag_local_identifier}) - Last Week")
        line.coords = list(zip(individual_data['location.long'], individual_data['location.lat']))

        # Set the color using the group color with alpha
        kml_color = 'b2' + group_color[1:]  # Add alpha to hex color
        line.style.linestyle.color = kml_color
        line.style.linestyle.width = 5  # Set line width

        # Set the timespan for the multiline
        line.timespan.begin = individual_data['timestamp'].min().strftime('%Y-%m-%dT%H:%M:%SZ')
        line.timespan.end = individual_data['timestamp'].max().strftime('%Y-%m-%dT%H:%M:%SZ')

# Save the KMZ file for the last week
kml.savekmz("plots\\kmls\\last_week.kmz")
print("KMZ file for the last week saved successfully.")

KMZ file for the last week saved successfully.


In [81]:
individual_data

,sensor_type_id,individual_local_identifier,barometric_pressure,data_decoding_software,eobs_activity,eobs_activity_samples,eobs_battery_voltage,eobs_fix_battery_voltage,eobs_horizontal_accuracy_estimate,eobs_key_bin_checksum,...,visible,geometry,azimuth,speed,tag_local_identifier,group_id,sex,location.long,location.lat,plot_name
3025550,653,24AD01_6R76,0,21,NaN,NaN,3996,3999,10.24,2404728320,...,True,36.8886144|0.4634593,1.522480,0.186143,10366,Periwinkle,m,36.888614,0.463459,Periwinkle
3025551,653,24AD01_6R76,0,21,NaN,NaN,3996,3996,14.34,2194916000,...,True,36.8887582|0.4634663,-0.573553,0.197067,10366,Periwinkle,m,36.888758,0.463466,Periwinkle
3025552,653,24AD01_6R76,0,21,NaN,NaN,3995,3999,32.51,992425123,...,True,36.8886443|0.4636438,0.450674,0.073820,10366,Periwinkle,m,36.888644,0.463644,Periwinkle
3025553,653,24AD01_6R76,0,21,NaN,NaN,3999,3995,17.66,2058152256,...,True,36.8886791|0.4637162,3.118670,0.088795,10366,Periwinkle,m,36.888679,0.463716,Periwinkle
3025554,653,24AD01_6R76,0,21,NaN,NaN,3996,3996,27.65,4168086981,...,True,36.8886813|0.4636196,1.373441,0.334784,10366,Periwinkle,m,36.888681,0.463620,Periwinkle
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3027428,653,24AD01_6R76,0,21,NaN,NaN,4000,4000,22.78,924332950,...,True,36.8888744|0.463406,-1.382618,0.302665,10366,Periwinkle,m,36.888874,0.463406,Periwinkle
3027429,653,24AD01_6R76,0,21,NaN,NaN,4000,4000,6.66,549937274,...,True,36.8885536|0.4634675,1.711761,0.093694,10366,Periwinkle,m,36.888554,0.463468,Periwinkle
3027430,653,24AD01_6R76,0,21,NaN,NaN,4000,3996,9.47,662463381,...,True,36.8886537|0.4634532,-1.130277,0.030257,10366,Periwinkle,m,36.888654,0.463453,Periwinkle
3027431,653,24AD01_6R76,0,21,NaN,NaN,3999,3999,8.45,3166809735,...,True,36.8886242|0.4634672,0.441582,0.023545,10366,Periwinkle,m,36.888624,0.463467,Periwinkle
